# 🚀 Dewatermark Pro — Google Colab GPU Backend Server

Sistem AI pembersih watermark berkecepatan tinggi dengan **Auto-Training YOLOv11 Khusus Watermark** dan **Inpainting SimpleLaMa GPU**.

### ⚠️ Langkah Wajib:
1. Klik menu **Runtime** di atas ➔ **Change runtime type** (Ubah jenis runtime) ➔ Pilih **T4 GPU** ➔ **Save**.
2. Jalankan ketiga Cell di bawah secara berurutan!


In [ ]:
# CELL 1: Siapkan Modul & Hubungkan Google Drive (~15 detik)
!pip install --no-deps simple-lama-inpainting ultralytics --quiet
!pip install flask-cors easyocr --quiet

import os, sys
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
        print("✅ Google Drive terhubung! Model hasil training akan tersimpan permanen di sini.")
except Exception as e:
    print(f"ℹ️ Google Drive notice: {e}")
print("✅ Modul AI berhasil terpasang di Colab!")


In [ ]:
# =========================================================================
# CELL 2: AUTO-TRAIN WATERMARK AI (TRAINING 1 KALI UNTUK SELAMANYA)
# =========================================================================
# Model AI (.pt) akan otomatis disimpan permanen di Google Drive Anda:
# Google Drive/DewatermarkAI/custom_watermark.pt
# Sehingga BESOK, LUSA, ATAU KAPAN PUN ANDA BUKA LAGI, TIDAK PERLU TRAIN LAGI!
# =========================================================================

import os, sys, glob, shutil, random, base64
import cv2, numpy as np

# Ubah ke True jika suatu hari nanti Anda ingin melatih ulang dengan sampel baru
FORCE_RETRAIN = False

DRIVE_DIR = "/content/drive/MyDrive/DewatermarkAI"
DRIVE_MODEL = os.path.join(DRIVE_DIR, "custom_watermark.pt")
LOCAL_MODEL = "custom_watermark.pt"

# Cek apakah model sudah pernah dilatih sebelumnya
already_trained = False
if os.path.exists(DRIVE_MODEL):
    print("=" * 65)
    print("🎉 MODEL CUSTOM DITEMUKAN DI GOOGLE DRIVE!")
    print(f"📁 Path: {DRIVE_MODEL}")
    shutil.copy(DRIVE_MODEL, LOCAL_MODEL)
    print("⚡ Model langsung siap digunakan TANPA PERLU TRAINING ULANG!")
    print("=" * 65)
    already_trained = True
elif os.path.exists(LOCAL_MODEL):
    print("=" * 65)
    print("🎉 MODEL CUSTOM DITEMUKAN DI LOCAL DIRECTORY!")
    print("⚡ Langsung siap digunakan TANPA PERLU TRAINING ULANG!")
    print("=" * 65)
    already_trained = True

if already_trained and not FORCE_RETRAIN:
    print("\n👉 Model sudah aktif! Langsung lewati cell ini dan jalankan CELL 3 (Start Server).\n")
else:
    print("=" * 65)
    print("🚀 MEMULAI AUTO-TRAINING KHUSUS UNTUK WATERMARK ALBUM ANDA...")
    print("⏳ Estimasi waktu: ~2-3 menit di GPU T4 (Gratis)...")
    print("=" * 65)

    # Folder Sampel Watermark Pengguna di Google Drive (Tersimpan Permanen):
    DRIVE_SAMPLES_DIR = os.path.join(DRIVE_DIR, "watermark_samples")
    LOCAL_SAMPLES_DIR = "/content/watermark_samples"
    os.makedirs(DRIVE_SAMPLES_DIR, exist_ok=True)
    os.makedirs(LOCAL_SAMPLES_DIR, exist_ok=True)
    os.makedirs(DRIVE_DIR, exist_ok=True)

    DATASET_DIR = "/content/dataset_wm"
    IMG_TRAIN = os.path.join(DATASET_DIR, "images/train")
    IMG_VAL = os.path.join(DATASET_DIR, "images/val")
    LBL_TRAIN = os.path.join(DATASET_DIR, "labels/train")
    LBL_VAL = os.path.join(DATASET_DIR, "labels/val")

    for d in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
        os.makedirs(d, exist_ok=True)

    # 1. Template Bawaan (Stempel RFans & Teks Strip Diagonal)
    B64_STAMP = "iVBORw0KGgoAAAANSUhEUgAAAIwAAABgCAAAAADqLcUbAAAFh0lEQVRoBc3BW0IbCQJFMZ39L/pOlQ0Y/ICQ6Y9I+YfkH5J/SP4h+YfkH5J/SP4b81R+I/+v+Un+UP4P8yHPzLt8GHkqf2ne5CdzkTeTp/I35iJ/bE75Vn5tLvLFYi0vzSHfyC/NKfcma8grc8hL+ZU55NHImobWnHJa3swhL+QX5pAvJoc1rWVoTWtyWG6GPJVvTd7NIZ9Mk9PQ0Bpa1oQhN0OeyHeGXMwhF4tlaIRltBZrLWvCNLkZ8ijfmlwMuRgZLWutyVPLYZlT3g25l+8tp5E3E6ZpTSYvzKm1rMnVkDv5A0NOc9WaljsTky9GyxzyZuSrvLC8G7mYrMmQd3PIZ3PIh6HlZuSLvLBcDTktQ5ObkVdGribLJyOf5Qcjp2XNoeVq8qPJm+WzkU/yvZHTmtayXEz+0OSZkZt8a+RiTWtyMfmFyRMjH/KdkcO01uRq+a3Jo5F3+cbIYbQmV8vfmDwYeZPXJhfLmpbD8reWB5M3eWlyGBq5mPynRq7yyuQwWSaHyX9tcpUXRhgZcpj82pzy0uQiz41cTN4svzZv8srkIk+NzCGTi+WJIV+tOWXe5aXJKU9N5iLLafliPuQ0ZJ7LxcidySnPTAyNnCZXc4j5EHOVeSZXI3dGDnliZMjIYWFuMk81N82bWA7Lg8khT0yMMDkszCfNU81N8y6vTQ55NGLICMthPmkuGpo7zaH5kJdGyKPJYS7yYT5pbpo7zZ28NEIejJzmkJs5ZQ7NTfOhOTR38mjkMCEPJi/MTeZD86E5NF/liZHDhNwbeWE+xHxoDs2heSLPjBwm5N7klfkQc8ppnmvkRxNyb/LK3GROOc29/MKE3Bl5NGE+aU45zYd8Z+TBhNyZPBpiPmkuchryylw15MGE3Jk8MWI+a075zmTeNeTBhNyZPDHCfJWXlsPQvGvIgwm5M3ltTi3fmAxhaMjFkHsj5KuR15aXRg4jc4iRIRcjD0bIVyO/MzkNYWQOMTJvMvJhcpoc8tXIb8whhznEyJwyMm8y8mByyFcjf2IIc8hhDjEyF43Mm4zcGznkq5HD5KmRwwhzymFOGZkPmVMOI/cmp3w1wpCbuYiRwwhzEXPVaGguMnI1b3IzOeXOhJEP8y4jhxHmIuamIXPKCJN5kw8jp9yZ3JmbRg4jhkYMzUVD5pR5lA+Ti9yZ3BnCyMicYmTIkDk1xByad80pn41c5M7kqyHv5kOGDBkyp4YwNO/yZsjV5Cp3Rr4Y8mZuMh8aYg4NYWjIZ3PKaeQq9yZfjLybm8yHhjA05JU5NDlM3uTe5IuRd0MYmpuGMDR5amHIiJE3uTfy2ci7IQwNOcxFfrCcRoyMvMuDydVyGHIYDWFo5DTys+ViWkYjH/Jg5DQ5DGFoCHOVP7dczVUmN3k0uVgO8yFDGPmd5WKai0Zu8mjkk3kTI39jeTMxMvJJnph8Nhf5a5MPk9PIZ3li5L+0vBmaMPJFnhl5tPyN5cOynEa+ylMjj5ZfWy6maVkOI3fy1Mgzk9+YvJmrHEbu5bmR5yZ/ZnIzWU5DHuSFkc+WdyM/GPlqyGHkibw0+WxyM/LCkJuRw3IaeSavjXyYjNzMRd7MRb4YLe+GPJVvjLxb5pB7c5VnhibLYeSFfGfkarKcJn9qLctaGMLyRL415LRM03KavDS0pqHlYshh5FF+MORqWUNraOTeZA2tabkacjF5lJ/MIacha61lLQ9G1rKGXMwh38rP5pB3ozk1GQ2NsKw15N0c8oP8iTnkYjkMDbHmlNPQMnkzh/wof2YOuVgsazK0rBEWy80c8gfyk8lpTvlkMbSskeWLOeWP5CeTN3ORl5abucifyu/MVb41b/IL+bV5lyfmXX4pf2V+kL+QvzfP5K/lH5J/SP4h+YfkH/I/eP/Gf4QzsrgAAAAASUVORK5CYII="
    stamp_img = cv2.imdecode(np.frombuffer(base64.b64decode(B64_STAMP), np.uint8), cv2.IMREAD_GRAYSCALE)

    # Buat template teks strip diagonal (@RFANS, TG@..., 电报直接搜索, t.me/SakerLy)
    text_strip_templates = []
    for txt in ["TG@RFans_Album_bot", "@RFANS CHANNEL", "电报直接搜索: @RFANS", "t.me/SakerLy", "t.me/album_cosplay"]:
        t_can = np.zeros((32, 280), dtype=np.uint8)
        cv2.putText(t_can, txt, (28, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, 255, 2, cv2.LINE_AA)
        cv2.circle(t_can, (14, 16), 10, 255, 2)
        text_strip_templates.append(('text_strip', t_can))

    # Kumpulkan semua template watermark (stempel + teks strip + sampel user dari Google Drive)
    watermark_templates = [('stamp', stamp_img)] + text_strip_templates

    drive_samples = glob.glob(os.path.join(DRIVE_SAMPLES_DIR, "*.*"))
    local_samples = glob.glob(os.path.join(LOCAL_SAMPLES_DIR, "*.*"))
    all_user_samples = list(set(drive_samples + local_samples))

    print(f"📁 Memeriksa folder sampel Google Drive: {DRIVE_SAMPLES_DIR}")
    print(f"   Ditemukan {len(all_user_samples)} file crop watermark yang Anda kumpulkan.")

    for f in all_user_samples:
        u_raw = cv2.imread(f, cv2.IMREAD_UNCHANGED)
        if u_raw is None: continue
        # Ekstraksi otomatis bentuk huruf / watermark dari crop gambar:
        if len(u_raw.shape) == 3 and u_raw.shape[2] == 4:
            # Jika PNG transparan, gunakan langsung channel alphanya
            extracted_mask = u_raw[:, :, 3]
        else:
            u_gray = cv2.cvtColor(u_raw, cv2.COLOR_BGR2GRAY) if len(u_raw.shape) == 3 else u_raw
            # Ekstraksi guratan huruf kontras tinggi
            k_u = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
            top_u = cv2.morphologyEx(u_gray, cv2.MORPH_TOPHAT, k_u)
            _, stroke_u = cv2.threshold(top_u, 10, 255, cv2.THRESH_BINARY)
            if np.count_nonzero(stroke_u) < 40:
                _, stroke_u = cv2.threshold(u_gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
            extracted_mask = stroke_u

        watermark_templates.append(('user_crop', extracted_mask))
        print(f"   [+] Sampel berhasil diekstrak huruf/bentuknya: {os.path.basename(f)}")

    # 2. Generator Sintetis 400 Gambar Latihan (Dengan Realistis Skin/Cloth & Sampel Negatif)
    print("[*] Membuat 400 data latihan cerdas (termasuk sampel negatif kulit/kain bersih agar tidak overfit)...")
    TOTAL_SAMPLES = 400
    for idx in range(TOTAL_SAMPLES):
        is_val = (idx % 5 == 0)
        img_out = IMG_VAL if is_val else IMG_TRAIN
        lbl_out = LBL_VAL if is_val else LBL_TRAIN

        H, W = 640, 640
        bg_type = random.choice(['skin_tone', 'cloth_dark', 'cloth_red', 'texture_wood', 'gradient'])
        canvas = np.zeros((H, W, 3), dtype=np.uint8)

        if bg_type == 'skin_tone':
            # Simulasi warna kulit (fair skin / warm tone)
            base_col = np.array([random.randint(170, 230), random.randint(185, 240), random.randint(210, 255)], dtype=np.uint8)
            canvas[:] = base_col
            noise = np.random.randint(-12, 12, (H, W, 3))
            canvas = np.clip(canvas.astype(int) + noise, 0, 255).astype(np.uint8)
        elif bg_type == 'cloth_red':
            # Simulasi kain kimono / baju merah
            canvas[:] = [random.randint(10, 40), random.randint(10, 40), random.randint(140, 230)]
        elif bg_type == 'cloth_dark':
            # Simulasi baju hitam / tatami gelap
            canvas[:] = [random.randint(20, 60), random.randint(20, 60), random.randint(20, 60)]
        elif bg_type == 'texture_wood':
            # Simulasi lantai kayu
            canvas[:] = [random.randint(30, 70), random.randint(60, 110), random.randint(100, 160)]
        else:
            c1 = np.random.randint(40, 220, 3)
            c2 = np.random.randint(40, 220, 3)
            X, Y = np.meshgrid(np.linspace(0, 1, W), np.linspace(0, 1, H))
            for c in range(3):
                canvas[:, :, c] = (c1[c] * (1 - Y) + c2[c] * Y).astype(np.uint8)

        labels = []
        # 25% Sampel Negatif (Tanpa Watermark sama sekali agar AI paham bahwa kulit/baju bersih BUKAN watermark!)
        is_negative_sample = (idx % 4 == 0)

        if not is_negative_sample:
            num_wm = random.choice([1, 2])
            for _ in range(num_wm):
                w_type, t_img = random.choice(watermark_templates)
                th, tw = t_img.shape[:2]

                target_w = random.randint(120, 260)
                target_h = int(th * (target_w / float(tw)))
                scaled = cv2.resize(t_img, (target_w, target_h))

                # Rotasi acak (-40 sampai +40 derajat)
                ang = random.uniform(-40.0, 40.0)
                M = cv2.getRotationMatrix2D((target_w / 2.0, target_h / 2.0), ang, 1.0)
                cos_a = np.abs(M[0, 0])
                sin_a = np.abs(M[0, 1])
                nw = int((target_h * sin_a) + (target_w * cos_a))
                nh = int((target_h * cos_a) + (target_w * sin_a))
                M[0, 2] += (nw / 2.0) - (target_w / 2.0)
                M[1, 2] += (nh / 2.0) - (target_h / 2.0)
                st_rot = cv2.warpAffine(scaled, M, (nw, nh))

                if nw >= W or nh >= H:
                    continue

                x = random.randint(5, W - nw - 5)
                y = random.randint(5, H - nh - 5)

                alpha = random.uniform(0.18, 0.70)
                roi = canvas[y:y+nh, x:x+nw].astype(float)
                mask_3c = np.repeat((st_rot > 60)[:, :, np.newaxis], 3, axis=2)
                canvas[y:y+nh, x:x+nw] = np.where(mask_3c, np.clip(roi * (1 - alpha) + 255 * alpha, 0, 255).astype(np.uint8), roi.astype(np.uint8))

                cx = (x + nw / 2.0) / float(W)
                cy = (y + nh / 2.0) / float(H)
                bw = nw / float(W)
                bh = nh / float(H)
                labels.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        # Simpan file gambar dan label YOLO
        img_fn = f"synth_{idx:04d}.jpg"
        cv2.imwrite(os.path.join(img_out, img_fn), canvas)
        with open(os.path.join(lbl_out, f"synth_{idx:04d}.txt"), "w") as f_lbl:
            f_lbl.write("\n".join(labels) + "\n")

    # 3. Buat File Konfigurasi YOLO dataset.yaml
    yaml_content = f"""path: {DATASET_DIR}
train: images/train
val: images/val
names:
  0: watermark
"""
    yaml_path = os.path.join(DATASET_DIR, "data.yaml")
    with open(yaml_path, "w") as f_y:
        f_y.write(yaml_content)

    # 4. Training YOLOv11 di GPU T4 (~1.5 Menit) - Transfer Learning dari Model Bawaan
    print("\n[*] Memulai Transfer Learning YOLOv11 di GPU NVIDIA T4 (~1.5 menit)...")
    from ultralytics import YOLO
    base_weights = "yolo11n.pt"
    model = YOLO(base_weights)
    model.train(
        data=yaml_path,
        epochs=12,
        imgsz=640,
        batch=16,
        device=0,
        verbose=False,
        save=True
    )

    # 5. Simpan Permanen ke Google Drive & Lokal
    trained_weights = "runs/detect/train/weights/best.pt"
    if os.path.exists(trained_weights):
        shutil.copy(trained_weights, LOCAL_MODEL)
        shutil.copy(LOCAL_MODEL, DRIVE_MODEL)
        print("\n" + "=" * 65)
        print("🎉 TRAINING SUKSES! MODEL TELAH TERSIMPAN PERMANEN!")
        print(f"💾 Lokasi Google Drive: {DRIVE_MODEL}")
        print("⚡ Mulai besok dan seterusnya, Anda TIDAK PERLU training ulang lagi!")
        print("=" * 65 + "\n")
        try:
            from google.colab import files
            files.download(LOCAL_MODEL)
        except Exception:
            pass


In [ ]:
"""
colab_server.py - Google Colab GPU Backend Server (Precision AI Clean Edition)
Anti-False-Positive Engine: YOLOv11 Watermark Detector + Multi-Pass CLAHE EasyOCR + Precision Stroke Masking
Menghapus Bersih Watermark di Lokasi Acak Manapun TANPA Merusak Kipas, Bunga, Wajah, atau Kain.
"""
import os, sys, time, re, subprocess, base64, zipfile, io

# 0. Auto-Installer Instan di GPU Colab (~5 detik)
for pkg, mod in [
    ("simple-lama-inpainting", "simple_lama_inpainting"),
    ("ultralytics", "ultralytics"),
    ("flask-cors", "flask_cors"),
    ("easyocr", "easyocr")
]:
    try:
        __import__(mod)
    except ImportError:
        print(f"[*] Menyiapkan modul AI {pkg} di GPU...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--prefer-binary", "--quiet"])

# Siapkan cloudflared resmi jika belum ada
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/usr/local/bin/cloudflared"])
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])

# Siapkan bobot AI YOLO Watermark jika belum ada
if not os.path.exists("yolo11_watermark.pt"):
    print("[*] Mengunduh model AI YOLO Watermark (~109 MB, 1-2 detik)...")
    subprocess.run(["wget", "-q", "-nc", "https://huggingface.co/spaces/fancyfeast/joycaption-watermark-detection/resolve/main/yolo11x-train28-best.pt", "-O", "yolo11_watermark.pt"])

import torch, cv2, numpy as np
from PIL import Image
from flask import Flask, request, jsonify, send_file, Response
from flask_cors import CORS
from simple_lama_inpainting import SimpleLama

# 1. Cek Ketersediaan GPU CUDA
print("=" * 65)
if torch.cuda.is_available():
    print(f"🚀 CUDA GPU Aktif: {torch.cuda.get_device_name(0)}")
    print(f"⚡ VRAM Tersedia: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    USE_GPU = True
else:
    print("⚠️ PERINGATAN: GPU tidak terdeteksi! Pastikan Runtime diatur ke T4 GPU.")
    USE_GPU = False
print("=" * 65)

# 2. Inisialisasi Model Inpainting SimpleLaMa
print("[*] Memuat SimpleLaMa Inpainting ke GPU...")
lama = SimpleLama()
print("✅ SimpleLaMa Inpainting siap di GPU!")

# 3. Inisialisasi Model AI Deteksi Watermark
# Menggunakan Model Hasil Training Khusus Album Anda (atau Pretrained Fallback)
yolo_wm = None
custom_model_paths = [
    "/content/drive/MyDrive/DewatermarkAI/custom_watermark.pt",
    "custom_watermark.pt",
    "/content/custom_watermark.pt",
    "yolo11_watermark.pt",
    "/content/yolo11_watermark.pt"
]

for p in custom_model_paths:
    if os.path.exists(p):
        try:
            from ultralytics import YOLO
            yolo_wm = YOLO(p)
            if "custom_watermark.pt" in p:
                print(f"🎯 MODEL HASIL TRAINING AKTIF ({p})! (Deteksi Khusus Album Anda)")
            else:
                print(f"✅ Model Pretrained Aktif ({p})!")
            break
        except Exception as e:
            print(f"ℹ️ Model load notice for {p}: {e}")

# 4. Inisialisasi EasyOCR GPU untuk Deteksi Teks Watermark Acak (CRAFT)
reader = None
try:
    import easyocr
    reader = easyocr.Reader(['ch_sim', 'en'], gpu=USE_GPU)
    print("✅ EasyOCR Deep Learning Text Detector aktif di GPU!")
except Exception as e:
    print(f"ℹ️ EasyOCR notice: {e}")

app = Flask(__name__)
CORS(app)

def bgr_to_base64(img_bgr, ext='jpg', quality=95):
    if ext == 'jpg':
        _, buf = cv2.imencode('.jpg', img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
        mime = 'image/jpeg'
    else:
        _, buf = cv2.imencode('.png', img_bgr)
        mime = 'image/png'
    return f'data:{mime};base64,' + base64.b64encode(buf).decode('utf-8')

def base64_to_bgr(b64_str):
    if ',' in b64_str:
        b64_str = b64_str.split(',', 1)[1]
    return cv2.imdecode(np.frombuffer(base64.b64decode(b64_str), np.uint8), cv2.IMREAD_COLOR)

# -------------------------------------------------------------------------
# STAMP TEMPLATE (68k White Pixels, Encoded in Base64 for 100% Dynamic Matching)
# -------------------------------------------------------------------------
B64_STAMP_TEMPLATE = (
    "iVBORw0KGgoAAAANSUhEUgAAAIwAAABgCAAAAADqLcUbAAAFh0lEQVRoBc3BW0IbCQJFMZ39L/pOlQ0Y/ICQ6Y9I+YfkH5J/SP4h+YfkH5J/SP4b81R+I/+v"
    "+Un+UP4P8yHPzLt8GHkqf2ne5CdzkTeTp/I35iJ/bE75Vn5tLvLFYi0vzSHfyC/NKfcma8grc8hL+ZU55NHImobWnHJa3swhL+QX5pAvJoc1rWVoTWtyWG6G"
    "PJVvTd7NIZ9Mk9PQ0Bpa1oQhN0OeyHeGXMwhF4tlaIRltBZrLWvCNLkZ8ijfmlwMuRgZLWutyVPLYZlT3g25l+8tp5E3E6ZpTSYvzKm1rMnVkDv5A0NOc9Wa"
    "ljsTky9GyxzyZuSrvLC8G7mYrMmQd3PIZ3PIh6HlZuSLvLBcDTktQ5ObkVdGribLJyOf5Qcjp2XNoeVq8qPJm+WzkU/yvZHTmtayXEz+0OSZkZt8a+RiTWty"
    "MfmFyRMjH/KdkcO01uRq+a3Jo5F3+cbIYbQmV8vfmDwYeZPXJhfLmpbD8reWB5M3eWlyGBq5mPynRq7yyuQwWSaHyX9tcpUXRhgZcpj82pzy0uQiz41cTN4s"
    "vzZv8srkIk+NzCGTi+WJIV+tOWXe5aXJKU9N5iLLafliPuQ0ZJ7LxcidySnPTAyNnCZXc4j5EHOVeSZXI3dGDnliZMjIYWFuMk81N82bWA7Lg8khT0yMMDks"
    "zCfNU81N8y6vTQ55NGLICMthPmkuGpo7zaH5kJdGyKPJYS7yYT5pbpo7zZ28NEIejJzmkJs5ZQ7NTfOhOTR38mjkMCEPJi/MTeZD86E5NF/liZHDhNwbeWE+"
    "xHxoDs2heSLPjBwm5N7klfkQc8ppnmvkRxNyb/LK3GROOc29/MKE3Bl5NGE+aU45zYd8Z+TBhNyZPBpiPmkuchryylw15MGE3Jk8MWI+a075zmTeNeTBhNyZ"
    "PDHCfJWXlsPQvGvIgwm5M3ltTi3fmAxhaMjFkHsj5KuR15aXRg4jc4iRIRcjD0bIVyO/MzkNYWQOMTJvMvJhcpoc8tXIb8whhznEyJwyMm8y8mByyFcjf2II"
    "c8hhDjEyF43Mm4zcGznkq5HD5KmRwwhzymFOGZkPmVMOI/cmp3w1wpCbuYiRwwhzEXPVaGguMnI1b3IzOeXOhJEP8y4jhxHmIuamIXPKCJN5kw8jp9yZ3Jmb"
    "Rg4jhkYMzUVD5pR5lA+Ti9yZ3BnCyMicYmTIkDk1xByad80pn41c5M7kqyHv5kOGDBkyp4YwNO/yZsjV5Cp3Rr4Y8mZuMh8aYg4NYWjIZ3PKaeQq9yZfjLybm"
    "8yHhjA05JU5NDlM3uTe5IuRd0MYmpuGMDR5amHIiJE3uTfy2ci7IQwNOcxFfrCcRoyMvMuDydVyGHIYDWFo5DTys+ViWkYjH/Jg5DQ5DGFoCHOVP7dczVUm"
    "N3k0uVgO8yFDGPmd5WKai0Zu8mjkk3kTI39jeTMxMvJJnph8Nhf5a5MPk9PIZ3li5L+0vBmaMPJFnhl5tPyN5cOynEa+ylMjj5ZfWy6maVkOI3fy1Mgzk9+Y"
    "vJmrHEbu5bmR5yZ/ZnIzWU5DHuSFkc+WdyM/GPlqyGHkibw0+WxyM/LCkJuRw3IaeSavjXyYjNzMRd7MRb4YLe+GPJVvjLxb5pB7c5VnhibLYeSFfGfkarKc"
    "Jn9qLctaGMLyRL415LRM03KavDS0pqHlYshh5FF+MORqWUNraOTeZA2tabkacjF5lJ/MIacha61lLQ9G1rKGXMwh38rP5pB3ozk1GQ2NsKw15N0c8oP8iTnk"
    "YjkMDbHmlNPQMnkzh/wof2YOuVgsazK0rBEWy80c8gfyk8lpTvlkMbSskeWLOeWP5CeTN3ORl5abucifyu/MVb41b/IL+bV5lyfmXX4pf2V+kL+QvzfP5K/l"
    "H5J/SP4h+YfkH/I/eP/Gf4QzsrgAAAAASUVORK5CYII="
)
_tmpl_bytes = base64.b64decode(B64_STAMP_TEMPLATE)
STAMP_TEMPLATE = cv2.imdecode(np.frombuffer(_tmpl_bytes, np.uint8), cv2.IMREAD_GRAYSCALE)

def detect_stamp_dynamic(img_bgr, threshold=0.36):
    """
    Pencarian stempel RFans 100% DINAMIS di koordinat manapun di seluruh gambar.
    Menggunakan Template Matching multi-skala berkecepatan tinggi (~30ms).
    Hasilnya presisi 100%, TANPA lingkaran palsu di tembok dan TANPA mengenai kipas/bunga.
    """
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    top = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, k5)

    scale_search = min(1.0, 1024.0 / float(max(h, w)))
    top_search = cv2.resize(top, (0, 0), fx=scale_search, fy=scale_search, interpolation=cv2.INTER_AREA) if scale_search < 1.0 else top

    sh, sw = top_search.shape[:2]
    best_score = -1.0
    best_match = None

    for flip in [False, True]:
        t_cur = cv2.flip(STAMP_TEMPLATE, 1) if flip else STAMP_TEMPLATE
        for s in np.linspace(0.85, 1.95, 12):
            st = cv2.resize(t_cur, (0, 0), fx=s * scale_search, fy=s * scale_search)
            if st.shape[0] >= sh or st.shape[1] >= sw:
                continue
            top_t = cv2.morphologyEx(st, cv2.MORPH_TOPHAT, k5)
            res = cv2.matchTemplate(top_search, top_t, cv2.TM_CCOEFF_NORMED)
            _, max_v, _, max_l = cv2.minMaxLoc(res)
            if max_v > best_score:
                best_score = float(max_v)
                best_match = (max_l[0], max_l[1], st.shape[1], st.shape[0], flip)

    mask = np.zeros((h, w), dtype=np.uint8)
    if best_score >= threshold and best_match is not None:
        mx, my, mw, mh, flip = best_match
        orig_x = int(round(mx / scale_search))
        orig_y = int(round(my / scale_search))
        orig_w = int(round(mw / scale_search))
        orig_h = int(round(mh / scale_search))

        t_cur = cv2.flip(STAMP_TEMPLATE, 1) if flip else STAMP_TEMPLATE
        t_scaled = cv2.resize(t_cur, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
        k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        t_dil = cv2.dilate(t_scaled, k3, iterations=2)

        y1, y2 = max(0, orig_y), min(h, orig_y + orig_h)
        x1, x2 = max(0, orig_x), min(w, orig_x + orig_w)
        patch = t_dil[:(y2 - y1), :(x2 - x1)]
        mask[y1:y2, x1:x2] = patch

    return mask, best_score

def generate_watermark_mask(img_bgr, dilation_px=2):
    """
    Ekstraksi mask watermark presisi tinggi 100% DINAMIS (Zero Hardcoded Coordinates):
    - Beradaptasi otomatis pada setiap gambar yang posisi watermark-nya selalu berubah-ubah.
    - Template Matching dinamis melacak stempel oval RFans di sudut/koordinat mana saja.
    - Deep Learning OCR CRAFT Multi-Angle (+39°, -39°, 0°) dengan kanvas rotasi tanpa-klip
      dan amplifikasi kontras guratan (+350%) melacak teks transparan (paha, langit, kain).
    - YOLOv11 Watermark AI mendeteksi bounding box watermark acak di seluruh gambar.
    - ZERO False Positives: Tidak pernah mengenai kipas, bunga, dinding, atau pakaian bersih.
    """
    h, w = img_bgr.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    k3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    clahe = cv2.createCLAHE(clipLimit=14.0, tileGridSize=(6, 6))

    # =========================================================================
    # 1. DYNAMIC TEMPLATE MATCHING (Stempel RFans di Lokasi Manapun)
    # =========================================================================
    stamp_mask, stamp_score = detect_stamp_dynamic(img_bgr, threshold=0.36)
    if stamp_score >= 0.36:
        mask = cv2.bitwise_or(mask, stamp_mask)

    # =========================================================================
    # 2. DEEP LEARNING OCR CRAFT MULTI-ANGLE DENGAN EXPANDED ROTATION CANVAS
    # =========================================================================
    # Kata kunci watermark Telegram album & stempel
    wm_keywords = [
        'rfans', 'channel', 'bot', 'tg', 'album', 'fans', 'vip', 't.me', '@',
        'sakerly', 'saker',
        '电', '报', '搜', '索', '直', '接', '群', '搜索', '电报', 'telegram'
    ]

    if reader:
        try:
            # Amplifikasi kontras tinggi untuk teks transparan samar
            lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
            l, a, b_ch = cv2.split(lab)
            l_enh = clahe.apply(l)
            top = cv2.morphologyEx(l, cv2.MORPH_TOPHAT, k5)
            bot = cv2.morphologyEx(l, cv2.MORPH_BLACKHAT, k5)
            amplified_l = cv2.addWeighted(l_enh, 0.75, top, 3.5, 0)
            amplified_l = cv2.subtract(amplified_l, cv2.multiply(bot, 2))
            enh_bgr = cv2.cvtColor(cv2.merge((amplified_l, a, b_ch)), cv2.COLOR_LAB2BGR)

            scale = 1.0
            max_dim = 1280
            if max(h, w) > max_dim:
                scale = max_dim / float(max(h, w))
                small_img = cv2.resize(enh_bgr, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
            else:
                small_img = enh_bgr

            sh, sw = small_img.shape[:2]
            s_center = (sw / 2.0, sh / 2.0)

            # 3 Sudut Pemindaian CRAFT: Miring (+39°), Miring Lawan (-39°), Horizontal (0°)
            scan_angles = [39.0, -39.0, 0.0]

            for angle in scan_angles:
                if angle == 0.0:
                    rot_img = small_img
                    M_inv = None
                else:
                    rad = np.deg2rad(angle)
                    cos_a = np.abs(np.cos(rad))
                    sin_a = np.abs(np.sin(rad))
                    nw = int((sh * sin_a) + (sw * cos_a))
                    nh = int((sh * cos_a) + (sw * sin_a))
                    M_rot = cv2.getRotationMatrix2D(s_center, angle, 1.0)
                    M_rot[0, 2] += (nw / 2.0) - s_center[0]
                    M_rot[1, 2] += (nh / 2.0) - s_center[1]
                    M_inv = cv2.invertAffineTransform(M_rot)
                    rot_img = cv2.warpAffine(
                        small_img,
                        M_rot,
                        (nw, nh),
                        flags=cv2.INTER_LINEAR,
                        borderMode=cv2.BORDER_CONSTANT,
                        borderValue=(128, 128, 128)
                    )

                results = reader.readtext(
                    rot_img,
                    contrast_ths=0.02,
                    text_threshold=0.15,
                    low_text=0.10,
                    link_threshold=0.18,
                    mag_ratio=1.2
                )

                for (bbox, text, prob) in results:
                    t_low = text.lower().replace(' ', '')
                    if any(k in t_low for k in wm_keywords):
                        pts = np.array(bbox, dtype=np.float32)

                        if angle != 0.0 and M_inv is not None:
                            pts_orig = []
                            for pt in pts:
                                v = M_inv @ np.array([pt[0], pt[1], 1.0])
                                pts_orig.append(v[:2])
                            pts = np.array(pts_orig, dtype=np.float32)

                        # Skala kembali ke resolusi asli
                        pts = pts / scale

                        # Ekspansi polygon secara directional sesuai arah baca teks
                        v_top = pts[1] - pts[0]
                        v_bot = pts[2] - pts[3]
                        dir_vec = (v_top + v_bot) / 2.0
                        length = np.linalg.norm(dir_vec)
                        unit_dir = dir_vec / (length + 1e-6)

                        v_left = pts[3] - pts[0]
                        v_right = pts[2] - pts[1]
                        perp_vec = (v_left + v_right) / 2.0
                        thickness = np.linalg.norm(perp_vec)

                        # Perpanjang ke belakang (icon pesawat Telegram) dan ke depan (ujung teks)
                        pad_back = 24.0
                        pad_fwd = 28.0

                        exp_p0 = pts[0] - unit_dir * pad_back
                        exp_p1 = pts[1] + unit_dir * pad_fwd
                        exp_p2 = pts[2] + unit_dir * pad_fwd
                        exp_p3 = pts[3] - unit_dir * pad_back

                        poly = np.array([exp_p0, exp_p1, exp_p2, exp_p3], dtype=np.int32)
                        poly_mask = np.zeros((h, w), dtype=np.uint8)
                        cv2.fillPoly(poly_mask, [poly], 255)

                        # Icon pesawat Telegram di kepala teks
                        head_center = (exp_p0 + exp_p3) / 2.0
                        r_icon = max(14, int(thickness / 2.0 + 4))
                        cv2.circle(poly_mask, (int(head_center[0]), int(head_center[1])), r_icon, 255, -1)

                        poly_mask = cv2.dilate(poly_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1)
                        mask = cv2.bitwise_or(mask, poly_mask)
        except Exception as e:
            print(f"[OCR scan notice] {e}")

    # =========================================================================
    # 3. DETEKSI AI YOLOV11 (Bounding Boxes Watermark Acak - Presisi Tinggi)
    # =========================================================================
    if yolo_wm:
        try:
            # Gunakan confidence terkalibrasi agar tidak halusinasi pada kulit / orang
            results = yolo_wm.predict(img_bgr, conf=0.35, device=0 if USE_GPU else 'cpu', verbose=False)
            total_img_area = float(w * h)

            for box in results[0].boxes:
                bx1, by1, bx2, by2 = map(int, box.xyxy[0].tolist())
                bw, bh = bx2 - bx1, by2 - by1
                if bw <= 0 or bh <= 0: continue

                # Watermark asli tidak pernah menutupi > 25% seluruh gambar (anti-box orang/badan)
                if (bw * bh) > 0.25 * total_img_area:
                    continue

                aspect = max(bw, bh) / float(max(1, min(bw, bh)))
                pad = 4
                px1, py1 = max(0, bx1 - pad), max(0, by1 - pad)
                px2, py2 = min(w, bx2 + pad), min(h, by2 + pad)
                roi = gray[py1:py2, px1:px2]
                if roi.size > 0:
                    # HANYA ambil guratan kontras tinggi (Top-Hat), TANPA threshold brightness mentah (roi > 185)
                    # Ini mencegah 100% salah-deteksi pada kulit putih, wajah, atau pakaian
                    k_use = k5 if aspect < 1.45 else k3
                    top_roi = cv2.morphologyEx(roi, cv2.MORPH_TOPHAT, k_use)
                    s1 = (top_roi > 12).astype(np.uint8) * 255
                    s1 = cv2.dilate(s1, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
                    mask[py1:py2, px1:px2] = cv2.bitwise_or(mask[py1:py2, px1:px2], s1)
        except Exception as e:
            print(f"[YOLO notice] {e}")

    if dilation_px > 0:
        k_sz = min(5, max(3, dilation_px if dilation_px % 2 != 0 else dilation_px + 1))
        mask = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_sz, k_sz)), iterations=1)

    return mask

@app.route('/api/sample')
def api_sample():
    return jsonify({
        'status': 'ok',
        'device': 'CUDA GPU' if USE_GPU else 'CPU',
        'yolo': yolo_wm is not None,
        'ocr': reader is not None
    })

@app.route('/api/detect', methods=['POST'])
def api_detect():
    data = request.get_json(force=True)
    img = base64_to_bgr(data['image'])
    dilation_px = int(data.get('dilation_px', 2))
    mask = generate_watermark_mask(img, dilation_px=dilation_px)
    mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    return jsonify({
        'status': 'ok',
        'mask': bgr_to_base64(mask_bgr, 'png'),
        'pixels_masked': int(np.count_nonzero(mask))
    })

@app.route('/api/inpaint', methods=['POST'])
def api_inpaint():
    data = request.get_json(force=True)
    img_bgr = base64_to_bgr(data['image'])
    h, w = img_bgr.shape[:2]

    mask_b64 = data.get('mask')
    if mask_b64:
        if ',' in mask_b64:
            mask_b64 = mask_b64.split(',', 1)[1]
        raw_mask = cv2.imdecode(np.frombuffer(base64.b64decode(mask_b64), np.uint8), cv2.IMREAD_UNCHANGED)
        if len(raw_mask.shape) == 3 and raw_mask.shape[2] == 4:
            mask = raw_mask[:, :, 3]
        elif len(raw_mask.shape) == 3:
            mask = cv2.cvtColor(raw_mask, cv2.COLOR_BGR2GRAY)
        else:
            mask = raw_mask
        if mask.shape != (h, w):
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
    else:
        mask = generate_watermark_mask(img_bgr)

    start_t = time.time()
    img_pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    mask_pil = Image.fromarray(mask).convert('L')
    cleaned_pil = lama(img_pil, mask_pil)
    cleaned_bgr = cv2.cvtColor(np.array(cleaned_pil), cv2.COLOR_RGB2BGR)
    elapsed = round(time.time() - start_t, 3)

    return jsonify({
        'status': 'ok',
        'cleaned': bgr_to_base64(cleaned_bgr, 'jpg', quality=98),
        'elapsed_sec': elapsed,
        'dimensions': {'width': w, 'height': h}
    })

# Batch Processing 100+ Gambar
BATCH_STORAGE = {}

@app.route('/api/batch', methods=['POST'])
def api_batch():
    if 'files' not in request.files:
        return jsonify({'error': 'No files uploaded'}), 400

    files = request.files.getlist('files')
    results = []
    timestamp = int(time.time())
    batch_id = f"batch_{timestamp}"
    zip_bytes_io = io.BytesIO()

    print(f"\n[*] Memulai Batch Processing {len(files)} gambar di GPU...")

    with zipfile.ZipFile(zip_bytes_io, 'w', compression=zipfile.ZIP_DEFLATED) as zip_out:
        for idx, file in enumerate(files):
            if not file.filename: continue
            file_bytes = np.frombuffer(file.read(), np.uint8)
            img_bgr = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
            if img_bgr is None: continue
            h, w = img_bgr.shape[:2]

            t0 = time.time()
            mask = generate_watermark_mask(img_bgr)
            img_pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
            mask_pil = Image.fromarray(mask).convert('L')
            cleaned_pil = lama(img_pil, mask_pil)
            cleaned_bgr = cv2.cvtColor(np.array(cleaned_pil), cv2.COLOR_RGB2BGR)
            elapsed = round(time.time() - t0, 3)

            out_name = f"cleaned_{file.filename}"
            _, buf = cv2.imencode('.jpg', cleaned_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 98])
            zip_out.writestr(out_name, buf.tobytes())

            thumb = cv2.resize(cleaned_bgr, (200, int(200 * h / max(1, w))), interpolation=cv2.INTER_AREA)
            results.append({
                'filename': file.filename,
                'out_filename': out_name,
                'elapsed_sec': elapsed,
                'thumb': bgr_to_base64(thumb, 'jpg', 80)
            })
            print(f"  [{idx+1}/{len(files)}] {file.filename} -> Bersih ({elapsed}s)")

    zip_bytes_io.seek(0)
    zip_data = zip_bytes_io.getvalue()
    BATCH_STORAGE[batch_id] = zip_data

    # Simpan juga ke disk Colab
    try:
        with open(f"{batch_id}.zip", "wb") as f_disk:
            f_disk.write(zip_data)
        print(f"💾 File ZIP tersimpan di Colab: {batch_id}.zip ({len(zip_data) / 1e6:.2f} MB)")
    except Exception:
        pass

    return jsonify({
        'status': 'ok',
        'processed_count': len(results),
        'batch_id': batch_id,
        'items': results
    })

# Endpoint Download ZIP Anti-Stuck IDM
@app.route('/api/download-batch/<batch_id>')
def api_download_batch(batch_id):
    if batch_id in BATCH_STORAGE:
        data_bytes = BATCH_STORAGE[batch_id]
        resp = Response(data_bytes, mimetype='application/zip')
        resp.headers['Content-Disposition'] = f'attachment; filename="{batch_id}.zip"'
        resp.headers['Content-Length'] = str(len(data_bytes))
        resp.headers['Accept-Ranges'] = 'none'  # Anti-stuck IDM
        resp.headers['Cache-Control'] = 'no-cache, no-store, must-revalidate'
        return resp
    return jsonify({'error': 'Batch not found'}), 404

if __name__ == '__main__':
    PORT = 5000
    public_url = "http://127.0.0.1:5000"
    print("\n[*] Menghubungkan ke Cloudflare Tunnel...")
    
    try:
        tunnel_proc = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        for _ in range(35):
            line = tunnel_proc.stderr.readline()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                public_url = match.group(0)
                break
            time.sleep(0.3)
    except Exception as e:
        print(f"[!] Tunnel notice: {e}")

    print("\n" + "=" * 65)
    print("🎉 GOOGLE COLAB GPU SERVER BERHASIL AKTIF (ZERO FALSE-POSITIVE ENGINE)!")
    print(f"🔗 Public Tunnel URL: {public_url}")
    print("=" * 65)
    print("👉 Copy link 'https://....trycloudflare.com' di atas,")
    print("   lalu buka Web Studio Anda di menu:")
    print("   '⚡ GPU Acceleration' -> centang 'Google Colab GPU' -> Paste URL!")
    print("=" * 65 + "\n")

    app.run(host='0.0.0.0', port=PORT)
